# LLM Webinar Series: Advanced LLM Capabilities

## Data Selection

### Chunking Strategy
The data used to demonstrate the chunking strategies in this notebook consists of the README.md and the README.pdf file for this project. In this section, we will walk through the following chunking strategies:
- Length-based
    - Character-based
    - Token-based
- Text-based
- Document-structure-based
    - Markdown
- Semantic-based

Under each subheading of each chunking strategy, the source of the example is provided.

#### Load the data

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

# Load the PDF
loader = PyPDFLoader("data/README.pdf")
pdf_documents = loader.load()

# Print the first page of the PDF used in the character-based and token-based chunking example
page_0 = pdf_documents[0].page_content
print(page_0)

#### Length-based


##### Character-based
Sources:  
How to split by character | 🦜️🔗 LangChain. https://python.langchain.com/docs/how_to/character_text_splitter/


In [ ]:
from langchain_text_splitters import CharacterTextSplitter

# Split Page 0 up to 100 characters with no overlap
text_splitter = CharacterTextSplitter(
    separator="", 
    chunk_size=100, 
    chunk_overlap=0
)
texts = text_splitter.split_text(page_0)

# For each chunk, print the character length and chunk content
for chunk in texts:
    print((len(chunk), chunk))

##### Token-based
Sources:  
How to split text by tokens | 🦜️🔗 LangChain. https://python.langchain.com/docs/how_to/split_by_token/


In [ ]:
import tiktoken
from langchain.text_splitter import TokenTextSplitter

# Split Page 0 up to 100 tokens with no overlap 
encoding = tiktoken.get_encoding("cl100k_base")
text_splitter = TokenTextSplitter(
    encoding_name="cl100k_base", 
    chunk_size=100, 
    chunk_overlap=0
)
texts = text_splitter.split_text(page_0)

# For each chunk, print the character length, token length, and chunk content
for chunk in texts:
    print((len(chunk), len(encoding.encode(chunk)), chunk))

#### Text-based
Sources:  
How to split by character | 🦜️🔗 LangChain. https://python.langchain.com/docs/how_to/character_text_splitter/

In [ ]:
# Split Page 0 up to 100 characters with no overlap based on the "\n" separator
text_splitter = CharacterTextSplitter(
    separator="\n", 
    chunk_size=100, 
    chunk_overlap=0
)
texts = text_splitter.split_text(page_0)

# For each chunk, print the character length and chunk content
for chunk in texts:
    print((len(chunk), len(encoding.encode(chunk)), chunk))

#### Document-Structure-Based 
##### Markdown
Sources:  
How to split Markdown by Headers | 🦜️🔗 LangChain. https://python.langchain.com/docs/how_to/markdown_header_metadata_splitter/  

In [ ]:
# Load the Markdown document
def load_md():
    with open("README.md", encoding="utf-8") as f:
        return f.read()
markdown_str = load_md()
print(markdown_str)

In [ ]:
from langchain_text_splitters import MarkdownHeaderTextSplitter

# Split the Markdown document based on headers
headers_to_split_on = [
    ("#", "Header 1"),
    ("##", "Header 2"),
]
splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=headers_to_split_on
)
documents = splitter.split_text(markdown_str)

# Print the split chunks
documents

#### Semantic Chunking
How to split text based on semantic similarity | 🦜️🔗 LangChain. https://python.langchain.com/docs/how_to/semantic-chunker/


In [ ]:
# Add PDF page contents to a list
pages = [page.page_content for page in pdf_documents]
pages

In [ ]:
# ❗ WARNING: DO NOT COMMIT YOUR API KEY❗ 
# 
# This cell is for local testing only. If you choose to set your
# API key here, it will be stored in plain text in the notebook file.
# 
# ❗ NEVER push or commit this notebook with your API key included. ❗
# 
# Recommended: Use a .env file (which is listed in .gitignore) to store your key.
# 
# If you are having issues setting up the OpenAI API key in the .env file,
# you may temporarily uncomment the code below and replace "YOUR-API-KEY".

# import os

# os.environ["OPENAI_API_KEY"] = "YOUR-API-KEY"

In [ ]:
import os
from langchain_experimental.text_splitter import SemanticChunker
from langchain_openai.embeddings import OpenAIEmbeddings

if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("OPENAI_API_KEY not found. Please create a .env file or set the variable manually.")

# Initialize the text splitter with OpenAI's "text-embedding-3-large" model
text_splitter = SemanticChunker(OpenAIEmbeddings(model="text-embedding-3-large"))

# Split the PDF document sematically
chunks = text_splitter.create_documents(pages)
chunks

## Evaluation

In [ ]:
import evaluate

reference_text =["The quick brown fox jumps over the lazy dog"]
predicted_text =["The hungry arctic fox jumps over the green frog"]

### BLEU
Sources:  
BLEU. In: Wikipedia. 2025. https://en.wikipedia.org/w/index.php?title=BLEU&oldid=1300846595   
Doshi K. Foundations of NLP Explained - Bleu Score and WER Metrics. Towards Data Science. May 9, 2021. https://towardsdatascience.com/foundations-of-nlp-explained-bleu-score-and-wer-metrics-1a5ba06d812b/

Metric:  
$$\text{BLEU} = \text{BP} \cdot \text{exp}\left(\sum_{n=1}^N w_n \text{log}(p_n)\right)$$

$\text{BP} = \text{Brevity penalty}$  
$n = \text{n-th gram}$  
$N = \text{n-grams considered}$  
$w_n = \text{weight of the n-th gram precision}$  
$p_n = \text{precision of predicted n-th gram} = \frac{\text{number of correctly predicted n-grams}}{\text{number of total predicted n-grams}}$

In [ ]:
bleu = evaluate.load("bleu")
print(bleu.compute(predictions=predicted_text, references=reference_text, max_order = 2))

### ROUGE
Sources:  
ROUGE (metric). In: Wikipedia. 2023. https://en.wikipedia.org/w/index.php?title=ROUGE_(metric)&oldid=1187242316  
Chiusano F. Two minutes NLP — Learn the ROUGE metric by examples. Generative AI. August 4, 2023. https://medium.com/nlplanet/two-minutes-nlp-learn-the-rouge-metric-by-examples-f179cc285499

Metric:
$$\text{ROUGE-N precision}=\frac{\text{number of correctly predicted n-grams}}{\text{number of total predicted n-grams}}$$
$$\text{ROUGE-N recall}=\frac{\text{number of correctly predicted n-grams}}{\text{number of total reference n-grams}}$$ 
$$\text{ROUGE-N F1}=\frac{2\cdot\text{ROUGE-N precision}\cdot\text{ROUGE-N recall}}{\text{ROUGE-N precision}+\text{ROUGE-N recall}}$$


In [ ]:
rouge = evaluate.load("rouge")
print(rouge.compute(predictions=predicted_text, references=reference_text))